# 09 — Four-Factor HRV Analysis

**Research question:** How do age, sex, blood pressure, and body mass index jointly relate to each individual HRV measurement, and which interactions are sufficiently supported to interpret?

This notebook extends the unifactorial, bifactorial, and trifactorial analyses (Notebooks 06–08) without treating these associations as causal. It includes complete original-category descriptive inventories; prespecified analysis-only categories; nested categorical models M1–M4; robust block interaction tests; conditional and marginal effects with uncertainty; four-way interaction estimability; continuous alternatives; distributional heterogeneity; and sensitivity analyses.

**Interpretation and guardrails.** In factorial models, the individual coefficient of a main effect does not represent a universal effect if interactions are present. Main effects are assessed in the additive model, pair interactions in M2, triplet interactions in M3, and the fourth-order interaction in M4. An interaction is not inferred from differences in subgroup p-values. HC3 uncertainty is not trusted if saturated cells produce leverage near 1. All stages keep the same 476 participants unless an explicitly labeled sensitivity subset is being analyzed.

**Outputs:** CSV tables in `outputs/09_four_factor_hrv_analysis/tables/`, and PNG figures arranged by **HRV variable** and analysis family. The input file is read-only. Code, methods and figure annotations use English throughout.

## 0 — Configuration, imported packages and cohort validation

In [1]:
from pathlib import Path
from itertools import combinations, product
from collections import Counter
import warnings
import numpy as np
import pandas as pd
import scipy.stats as st
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import quantreg
from statsmodels.stats.multitest import multipletests
from patsy import dmatrix, build_design_matrices
from sklearn.model_selection import KFold
from IPython.display import display

INPUT_FILE = Path('../../outputs/dataset_final.csv')
OUT = Path('../../eda_results/09_four_factor_hrv_analysis')
TAB = OUT / 'tables'
FIG = OUT / 'figures'
GLOBAL = FIG / '_global'
for directory in (TAB, FIG, GLOBAL):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 20261125
MIN_CELL_N = 5                     # interpretation threshold for original/fine-grained cells
MIN_CONTRAST_N = 10               # comparison eligibility of two sampled groups
MAX_STABLE_LEVERAGE = 0.95        # conservative HC3 influence warning threshold
N_BOOT = 150                      # sensitivity / uncertainty bootstrap resamples
CV_FOLDS = 5

HRV = {
    't2m_pre_mean_rr': ('mean_rr', 'Mean RR', 'ms'),
    't2m_pre_mean_hr': ('mean_hr', 'Mean HR', 'bpm'),
    't2m_pre_sdnn': ('sdnn', 'SDNN', 'ms'),
    't2m_pre_rmssd': ('rmssd', 'RMSSD', 'ms'),
    't2m_pre_hf': ('hf', 'HF', 'ms²'),
    't2m_pre_lf': ('lf', 'LF', 'ms²'),
    't2m_pre_vlf': ('vlf', 'VLF', 'ms²'),
}
FACTORS = {
    'age_group': ['60-69', '70-79', '80+'],
    'sex_category': ['Female', 'Male'],
    'bp_category': ['Optimal', 'Normal', 'High-normal', 'Stage 1', 'Stage 2', 'Stage 3'],
    'bmi_category': ['Underweight', 'Normal', 'Overweight', 'Obesity'],
}
FACTOR_KEYS = list(FACTORS)
SHORT = {'age_group':'age','sex_category':'sex','bp_category':'bp','bmi_category':'bmi'}
FACTOR_LABELS = {'age_group':'Age','sex_category':'Sex','bp_category':'Blood pressure','bmi_category':'BMI'}
ANA = {'age_group':'age_analysis','sex_category':'sex_category','bp_category':'bp_analysis','bmi_category':'bmi_analysis'}
LEVELS = {
    'age_group':['60-69','70+'],
    'sex_category':['Female','Male'],
    'bp_category':['Optimal/Normal','High-normal','Stage 1-3'],
    'bmi_category':['Underweight/Normal','Overweight','Obesity'],
}
ANA_KEYS = [ANA[f] for f in FACTOR_KEYS]
COLORS = {
    'age_group':dict(zip(LEVELS['age_group'],sns.color_palette('viridis',2))),
    'sex_category':dict(zip(LEVELS['sex_category'],sns.color_palette('colorblind',2))),
    'bp_category':dict(zip(LEVELS['bp_category'],sns.color_palette('crest',3))),
    'bmi_category':dict(zip(LEVELS['bmi_category'],sns.color_palette('flare',3))),
}
for col,(slug,_,_) in HRV.items():
    for branch in ('01_descriptive','02_model_hierarchy','03_pair_interactions','04_adjusted_effects',
                   '05_higher_interactions','06_continuous','07_diagnostics'):
        (FIG / slug / branch).mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid',context='notebook',font_scale=1.02)
plt.rcParams.update({'figure.facecolor':'white','savefig.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'grid.alpha':.18,
    'axes.titleweight':'semibold','savefig.dpi':170})

df = pd.read_csv(INPUT_FILE)
required = list(HRV) + FACTOR_KEYS + ['age','imc','bp_systolic','bp_diastolic']
missing = sorted(set(required)-set(df.columns))
if missing:
    raise ValueError(f'Final dataset is missing columns: {missing}')
if df[required].isna().any().any():
    raise ValueError('Final dataset has missing HRV/factor values; review preprocessing upstream')
if not all(pd.api.types.is_numeric_dtype(df[c]) for c in list(HRV)+['age','imc','bp_systolic','bp_diastolic']):
    raise TypeError('Expected numeric outcomes and continuous factors')
for f,options in FACTORS.items():
    other = set(df[f].unique()) - set(options)
    if other:
        raise ValueError(f'Unexpected values for {f}: {other}')

# Analysis-specific grouping (identical to Notebook 08); original columns are preserved.
df['age_analysis']=df['age_group'].replace({'70-79':'70+','80+':'70+'})
df['bp_analysis']=df['bp_category'].replace({'Optimal':'Optimal/Normal','Normal':'Optimal/Normal',
    'Stage 1':'Stage 1-3','Stage 2':'Stage 1-3','Stage 3':'Stage 1-3'})
df['bmi_analysis']=df['bmi_category'].replace({'Underweight':'Underweight/Normal','Normal':'Underweight/Normal'})
for f in FACTOR_KEYS:
    df[ANA[f]]=pd.Categorical(df[ANA[f]],categories=LEVELS[f],ordered=True)

SCALES={}
for col in HRV:
    v=df[col].to_numpy(float)
    SCALES[col]='log10' if np.min(v)>0 and np.quantile(v,.99)/max(np.quantile(v,.01),1e-12)>=30 else 'identity'
scale_table=pd.DataFrame([{'hrv_variable':key,'hrv_label':val[1],'model_outcome_scale':SCALES[key],
    'minimum':df[key].min(),'q01':df[key].quantile(.01),'q99':df[key].quantile(.99)} for key,val in HRV.items()])
scale_table.to_csv(TAB/'outcome_scales.csv',index=False)
assert len(df)==476, f'Expected validated final cohort (476), found {len(df)}'
print(f'Validated final cohort: {len(df)} participants | {len(HRV)} HRV outcomes | four factors')
print('All figure folders organized first by individual HRV outcome.')
display(scale_table)

Validated final cohort: 476 participants | 7 HRV outcomes | four factors
All figure folders organized first by individual HRV outcome.


,hrv_variable,hrv_label,model_outcome_scale,minimum,q01,q99
0,t2m_pre_mean_rr,Mean RR,identity,468.0,586.000,1129.25
1,t2m_pre_mean_hr,Mean HR,identity,46.0,53.000,102.50
2,t2m_pre_sdnn,SDNN,log10,4.8,6.725,355.75
3,t2m_pre_rmssd,RMSSD,log10,0.5,1.275,43.50
4,t2m_pre_hf,HF,log10,1.0,3.000,5772.50
5,t2m_pre_lf,LF,log10,7.0,10.750,6915.00
6,t2m_pre_vlf,VLF,log10,1.0,2.000,1452.50


## 1 — Four-way support and descriptive distribution inventory

The original factor grid has **3 × 2 × 6 × 4 = 144 possible cells**. The analysis grid has **2 × 2 × 3 × 3 = 36**. Every possible cell is represented in the exported inventory, including empty cells, and every observed category remains visible in descriptive outputs. A nonempty cell is not automatically reliable for inference.

In [2]:
def full_grid(scheme):
    fs=FACTOR_KEYS if scheme=='original' else ANA_KEYS
    cats=FACTORS if scheme=='original' else {ANA[f]:LEVELS[f] for f in FACTOR_KEYS}
    skeleton=pd.MultiIndex.from_product([cats[k] for k in fs],names=fs).to_frame(index=False)
    grouped=df.groupby(fs,observed=True).size().rename('n').reset_index()
    out=skeleton.merge(grouped,on=fs,how='left')
    out['n']=out['n'].fillna(0).astype(int)
    out['scheme']=scheme
    out['cell_empty']=out['n'].eq(0)
    out['cell_lt5']=out['n'].lt(5)
    out['cell_lt10']=out['n'].lt(10)
    out=out.rename(columns=dict(zip(fs,['age_level','sex_level','bp_level','bmi_level'])))
    return out
support=pd.concat([full_grid('original'),full_grid('analysis')],ignore_index=True)
support.to_csv(TAB/'four_factor_cell_counts.csv',index=False)
support_summary=(support.groupby('scheme',as_index=False)
    .agg(cells=('n','size'),occupied=('cell_empty',lambda v:int((~v).sum())),
         empty=('cell_empty','sum'),cells_lt5=('cell_lt5','sum'),cells_lt10=('cell_lt10','sum'),
         minimum_n=('n','min'),total_participants=('n','sum')))
support_summary.to_csv(TAB/'four_factor_support_summary.csv',index=False)
assert support_summary.total_participants.eq(476).all()

# Both original and collapsed definitions are described; no participants are dropped.
desc=[]
for scheme,fs in [('original',FACTOR_KEYS),('analysis',ANA_KEYS)]:
    for keys,g in df.groupby(fs,observed=True):
        group=dict(zip(['age_level','sex_level','bp_level','bmi_level'],keys))
        for col,(slug,label,unit) in HRV.items():
            v=g[col].to_numpy(float)
            desc.append({'scheme':scheme,**group,'hrv_variable':col,'hrv_label':label,'n':len(v),
                'mean':np.mean(v),'median':np.median(v),'sd':np.std(v,ddof=1) if len(v)>1 else np.nan,
                'q05':np.quantile(v,.05),'q25':np.quantile(v,.25),'q75':np.quantile(v,.75),
                'q95':np.quantile(v,.95),'min':np.min(v),'max':np.max(v),
                'iqr':np.quantile(v,.75)-np.quantile(v,.25),'small_n_warning':len(v)<MIN_CELL_N})
descriptive=pd.DataFrame(desc)
descriptive.to_csv(TAB/'four_factor_hrv_descriptives.csv',index=False)

# Four-panel BP × BMI support atlas conditioned on age and sex.
for scheme in ('original','analysis'):
    s=support.loc[support.scheme==scheme]
    ages=FACTORS['age_group'] if scheme=='original' else LEVELS['age_group']
    sexes=FACTORS['sex_category']
    bp=FACTORS['bp_category'] if scheme=='original' else LEVELS['bp_category']
    bmi=FACTORS['bmi_category'] if scheme=='original' else LEVELS['bmi_category']
    fig,axes=plt.subplots(len(ages),2,figsize=(12.5,4.4*len(ages)),squeeze=False)
    for ai,age in enumerate(ages):
        for si,sex in enumerate(sexes):
            sub=s[(s.age_level==age)&(s.sex_level==sex)]
            mat=sub.pivot(index='bp_level',columns='bmi_level',values='n').reindex(index=bp,columns=bmi)
            sns.heatmap(mat,ax=axes[ai,si],cmap='Blues',annot=True,fmt='d',linewidths=.6,
                linecolor='white',annot_kws={'fontsize':10},cbar=False)
            axes[ai,si].set(title=f'Age {age} | {sex}',xlabel='BMI',ylabel='Blood pressure')
            axes[ai,si].tick_params(axis='x',rotation=24,labelsize=9)
    fig.suptitle(f'Four-factor support | {scheme} categories',y=.998,fontsize=17)
    fig.tight_layout(rect=(0,0,1,.974),h_pad=3)
    fig.savefig(GLOBAL/f'four_factor_support_{scheme}.png',bbox_inches='tight',dpi=170)
    plt.close(fig)
print('Original and analysis grids:',support_summary.to_dict('records'))
display(support_summary)

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Original and analysis grids: [{'scheme': 'analysis', 'cells': 36, 'occupied': 36, 'empty': 0, 'cells_lt5': 3, 'cells_lt10': 14, 'minimum_n': 1, 'total_participants': 476}, {'scheme': 'original', 'cells': 144, 'occupied': 93, 'empty': 51, 'cells_lt5': 109, 'cells_lt10': 128, 'minimum_n': 0, 'total_participants': 476}]


,scheme,cells,occupied,empty,cells_lt5,cells_lt10,minimum_n,total_participants
0,analysis,36,36,0,3,14,1,476
1,original,144,93,51,109,128,0,476


## 2 — Hierarchical factorial models M1–M4 and robust inference

All nested models use **sum-to-zero contrasts**. HC3 Wald tests are used only when the design matrix is full rank and the covariance is finite with acceptable leverage. At M4, any singleton cell may make the saturated model non-robust; such results are explicitly marked `unstable_hc3`, while descriptive model fit is still recorded. Partial R² is computed from the reduction in residual sum of squares for each nested model; this is a descriptive incremental fit quantity, not evidence of predictive generalization. Cross-validation uses a fixed design basis and checks training rank in every fold.

In [3]:
TERMS={f:f'C({ANA[f]}, Sum)' for f in FACTOR_KEYS}
TERM_BLOCKS={k:[':'.join(TERMS[x] for x in combo) for combo in combinations(FACTOR_KEYS,k)]
             for k in (1,2,3,4)}
STAGES={'M1':1,'M2':2,'M3':3,'M4':4}
ALL_LEVELS=[tuple(x) for x in product(*(LEVELS[f] for f in FACTOR_KEYS))]

# Store the actual matrices and Patsy design metadata; do not access statsmodels' data.design_info.
DESIGNS={}
for stage,order in STAGES.items():
    rhs=' + '.join(t for k in range(1,order+1) for t in TERM_BLOCKS[k])
    x=dmatrix(rhs,df,return_type='dataframe')
    DESIGNS[stage]={'matrix':x,'info':x.design_info,'rhs':rhs,
        'block_terms':TERM_BLOCKS[order],'columns':list(x.columns)}
    assert np.linalg.matrix_rank(x.to_numpy(float))==x.shape[1],f'{stage} is rank deficient'


def outcome(col,scale=None,data=None):
    subset=df if data is None else data
    scale=SCALES[col] if scale is None else scale
    v=subset[col].to_numpy(dtype=float)
    if scale=='log10':
        if not np.all(v>0):
            raise ValueError(f'{col} has nonpositive values; cannot use log10')
        return np.log10(v)
    return v


def fit_robust(y,X):
    model=sm.OLS(y,np.asarray(X,dtype=float)).fit(cov_type='HC3')
    hat=np.asarray(model.get_influence().hat_matrix_diag)
    cov=np.asarray(model.cov_params())
    reliable=(np.linalg.matrix_rank(np.asarray(X,dtype=float))==X.shape[1]
              and np.isfinite(cov).all() and np.max(hat)<MAX_STABLE_LEVERAGE)
    return model,{'rank':int(model.model.rank),'n_parameters':int(X.shape[1]),
        'max_leverage':float(hat.max()),'n_leverage_ge_95':int((hat>=MAX_STABLE_LEVERAGE).sum()),
        'condition_number':float(model.condition_number),'robust_inference_ok':bool(reliable)}


def wald_term(model,info,term):
    if term not in info.term_name_slices:
        return np.nan
    sub=info.term_name_slices[term]
    selectors=np.arange(sub.start,sub.stop)
    R=np.eye(len(info.column_names))[selectors]
    try:
        test=model.wald_test(R,scalar=True)
        p=float(test.pvalue)
        return p if np.isfinite(p) else np.nan
    except (ValueError,np.linalg.LinAlgError,ZeroDivisionError):
        return np.nan


def wald_block(model,info,terms):
    indices=[]
    for term in terms:
        if term not in info.term_name_slices:
            return np.nan
        sl=info.term_name_slices[term]
        indices.extend(range(sl.start,sl.stop))
    try:
        p=float(model.wald_test(np.eye(len(info.column_names))[indices],scalar=True).pvalue)
        return p if np.isfinite(p) else np.nan
    except (ValueError,np.linalg.LinAlgError,ZeroDivisionError):
        return np.nan


def bh(table,p_name,q_name):
    out=table.copy()
    out[q_name]=np.nan
    if len(out) and p_name in out:
        good=np.isfinite(pd.to_numeric(out[p_name],errors='coerce'))
        if good.any():
            out.loc[good,q_name]=multipletests(out.loc[good,p_name].astype(float),method='fdr_bh')[1]
    return out


def cv_model(y,X,seed):
    kf=KFold(n_splits=CV_FOLDS,shuffle=True,random_state=seed)
    errors=[];ranks=[]
    for train,test in kf.split(X):
        xt=X[train];xv=X[test]
        rank=np.linalg.matrix_rank(xt)
        ranks.append(rank==X.shape[1])
        if rank!=X.shape[1]:
            continue
        beta=np.linalg.lstsq(xt,y[train],rcond=None)[0]
        errors.append(float(np.mean((y[test]-xv@beta)**2)))
    return {'cv_rmse_model_scale':float(np.sqrt(np.mean(errors))) if errors else np.nan,
            'cv_valid_folds':len(errors),'cv_total_folds':CV_FOLDS,
            'cv_full_rank_folds':int(sum(ranks))}

hierarchy_records=[];term_records=[];FITS={}
for hi,col in enumerate(HRV):
    y=outcome(col)
    prev_rss=None
    for stage,order in STAGES.items():
        d=DESIGNS[stage]
        X=d['matrix'].to_numpy(float)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',RuntimeWarning)
            fit,diag=fit_robust(y,X)
        FITS[(col,stage)]=fit
        rss=float(np.sum(fit.resid**2))
        base={ 'hrv_variable':col,'hrv_label':HRV[col][1], 'model_stage':stage,
            'outcome_scale':SCALES[col],'n':len(y),'r2':float(fit.rsquared),
            'adjusted_r2':float(fit.rsquared_adj),'aic':float(fit.aic),'bic':float(fit.bic),
            'rss':rss,'added_block':{1:'main effects',2:'two-way interactions',
            3:'three-way interactions',4:'four-way interaction'}[order],
            'incremental_partial_r2':max(0,(prev_rss-rss)/prev_rss) if prev_rss is not None and prev_rss>0 else np.nan,
            **diag}
        base['model_status']='ok' if diag['robust_inference_ok'] else 'unstable_hc3'
        base['added_block_p_hc3']=wald_block(fit,d['info'],d['block_terms']) if diag['robust_inference_ok'] else np.nan
        base.update(cv_model(y,X,SEED+hi))
        hierarchy_records.append(base)
        if diag['robust_inference_ok']:
            for term in d['block_terms']:
                term_records.append({'hrv_variable':col,'model_stage':stage,'interaction_order':order,
                    'term':term,'factors':' × '.join(SHORT[f] for f in FACTOR_KEYS if TERMS[f] in term),
                    'df_term':d['info'].term_name_slices[term].stop-d['info'].term_name_slices[term].start,
                    'p_hc3':wald_term(fit,d['info'],term),
                    'model_scale':SCALES[col]})
        prev_rss=rss
hierarchy=bh(pd.DataFrame(hierarchy_records),'added_block_p_hc3','added_block_q_bh')
term_effects=bh(pd.DataFrame(term_records),'p_hc3','term_q_bh_all')
hierarchy.to_csv(TAB/'hierarchical_models.csv',index=False)
term_effects.to_csv(TAB/'individual_effect_terms.csv',index=False)
assert len(hierarchy)==len(HRV)*4
assert all((hierarchy.groupby('hrv_variable').nunique()['model_stage']==4))
print('Model hierarchy status:')
display(hierarchy.groupby(['model_stage','model_status']).size().rename('n').to_frame())

Model hierarchy status:


,,n
model_stage,model_status,
M1,ok,7
M2,ok,7
M3,ok,7
M4,unstable_hc3,7


## 3 — Standardized adjusted effects and conditional pair contrasts

Marginal contrasts from M1 are standardized over the complete empirical distribution of the other factors. Pair-conditional contrasts from M2 are standardized over the observed distribution of the *remaining* factors within the conditioning level, with a minimum observed group size and explicit common-support check. Confidence intervals use the model's HC3 covariance; p-values undergo Benjamini–Hochberg correction within their stated family. This is **associational standardization**, not causal adjustment or an intervention.

In [4]:
def matrix_for(stage,newdata):
    design=DESIGNS[stage]
    mat=build_design_matrices([design['info']],newdata,return_type='dataframe')[0]
    assert list(mat.columns)==design['columns'],'Design columns changed unexpectedly'
    return mat.to_numpy(dtype=float)


def delta_from_design(model,Xa,Xb):
    # Xa/Xb have the same rows and refer to alternative factor assignments.
    v=(Xa-Xb).mean(axis=0)
    est=float(v@model.params)
    vcov=np.asarray(model.cov_params(),float)
    se=float(np.sqrt(max(0.,v@vcov@v)))
    z=est/se if se>0 else np.nan
    p=float(2*st.norm.sf(abs(z))) if np.isfinite(z) else np.nan
    return est,est-1.9599639845*se,est+1.9599639845*se,p


def matched_rows(focal,levels,condition_on=None,level=None):
    data=df if condition_on is None else df.loc[df[ANA[condition_on]]==level]
    x1=data[ANA_KEYS].copy()
    x2=x1.copy()
    x1[ANA[focal]]=levels[0]
    x2[ANA[focal]]=levels[1]
    # Reapply the exact categorical universe, including non-observed factor levels.
    for f in FACTOR_KEYS:
        key=ANA[f]
        x1[key]=pd.Categorical(x1[key],categories=LEVELS[f])
        x2[key]=pd.Categorical(x2[key],categories=LEVELS[f])
    return x1,x2


def common_support(focal,lv1,lv2,conditioning=None,lv_conditioning=None):
    keys=[f for f in FACTOR_KEYS if f!=focal and f!=conditioning]
    basis=df if conditioning is None else df.loc[df[ANA[conditioning]]==lv_conditioning]
    eligible=basis.groupby([ANA[f] for f in keys],observed=True).size().reset_index(name='n')
    # Strict overlap: cell in each focal level exists for every observed background combination.
    group=df if conditioning is None else df.loc[df[ANA[conditioning]]==lv_conditioning]
    rates=[]
    for value in (lv1,lv2):
        counts=(group.loc[group[ANA[focal]]==value]
                .groupby([ANA[f] for f in keys],observed=True).size().reset_index(name='match_n'))
        merged=eligible.merge(counts,on=[ANA[f] for f in keys],how='left')
        rates.append(float(merged.loc[merged.match_n.fillna(0)>0,'n'].sum()/max(merged.n.sum(),1)))
    return min(rates)

marginal_rows=[]
for col in HRV:
    fit=FITS[col,'M1']
    okay=bool(hierarchy.loc[(hierarchy.hrv_variable==col)&(hierarchy.model_stage=='M1'),'robust_inference_ok'].iloc[0])
    for f in FACTOR_KEYS:
        for a,b in combinations(LEVELS[f],2):
            r={'hrv_variable':col,'factor':f,'level_1':a,'level_2':b,'outcome_scale':SCALES[col],
               'method':'M1 standardized HC3','model_robust_ok':okay}
            x1,x2=matched_rows(f,(a,b))
            if okay:
                est,low,high,p=delta_from_design(fit,matrix_for('M1',x1),matrix_for('M1',x2))
                r.update(estimate=est,ci95_lower=low,ci95_upper=high,p_hc3=p)
            marginal_rows.append(r)
marginal=bh(pd.DataFrame(marginal_rows),'p_hc3','q_bh_all_adjusted_marginals')
marginal.to_csv(TAB/'adjusted_marginal_effects.csv',index=False)

conditional_rows=[]
nonparam_rows=[]
for hi,col in enumerate(HRV):
    fit=FITS[col,'M2']
    okay=bool(hierarchy.loc[(hierarchy.hrv_variable==col)&(hierarchy.model_stage=='M2'),'robust_inference_ok'].iloc[0])
    for pair in combinations(FACTOR_KEYS,2):
        for focal,conditioning in (pair,pair[::-1]):
            for cond_level in LEVELS[conditioning]:
                for a,b in combinations(LEVELS[focal],2):
                    xa=df.loc[(df[ANA[focal]]==a)&(df[ANA[conditioning]]==cond_level),col].to_numpy(float)
                    xb=df.loc[(df[ANA[focal]]==b)&(df[ANA[conditioning]]==cond_level),col].to_numpy(float)
                    eligible=min(len(xa),len(xb))>=MIN_CONTRAST_N
                    overlap=common_support(focal,a,b,conditioning,cond_level)
                    supported=eligible and overlap>=.95
                    base={'hrv_variable':col,'pair':'_x_'.join(SHORT[f] for f in pair),
                        'focal_factor':focal,'conditioning_factor':conditioning,
                        'conditioning_level':cond_level,'focal_level_1':a,'focal_level_2':b,
                        'n_1':len(xa),'n_2':len(xb),'min_n_required':MIN_CONTRAST_N,
                        'common_support_fraction':overlap,'model_scale':SCALES[col],
                        'status':'ok' if supported and okay else ('small_n' if not eligible else 'insufficient_common_support' if overlap<.95 else 'unstable_model')}
                    record=base.copy()
                    if supported and okay:
                        x1,x2=matched_rows(focal,(a,b),conditioning,cond_level)
                        est,low,high,p=delta_from_design(fit,matrix_for('M2',x1),matrix_for('M2',x2))
                        record.update(estimate=est,ci95_lower=low,ci95_upper=high,p_hc3=p)
                    conditional_rows.append(record)
                    nr=base.copy()
                    if eligible:
                        # Mann–Whitney's U gives the probability-of-superiority effect exactly.
                        u,p=st.mannwhitneyu(xa,xb,alternative='two-sided')
                        delta=float(2*u/(len(xa)*len(xb))-1)
                        rng=np.random.default_rng(SEED+10000*hi+len(nonparam_rows))
                        boot=np.empty(N_BOOT)
                        for j in range(N_BOOT):
                            a_star=xa[rng.integers(len(xa),size=len(xa))]
                            b_star=xb[rng.integers(len(xb),size=len(xb))]
                            u_boot=st.mannwhitneyu(a_star,b_star).statistic
                            boot[j]=2*u_boot/(len(a_star)*len(b_star))-1
                        nr.update(cliffs_delta=delta,ci95_lower=float(np.quantile(boot,.025)),
                            ci95_upper=float(np.quantile(boot,.975)),mw_p=float(p),
                            median_difference=float(np.median(xa)-np.median(xb)),
                            ks_statistic=float(st.ks_2samp(xa,xb).statistic),
                            wasserstein=float(st.wasserstein_distance(xa,xb)),
                            bf_p=float(st.levene(xa,xb,center='median').pvalue))
                    nonparam_rows.append(nr)
conditional=bh(pd.DataFrame(conditional_rows),'p_hc3','q_bh_conditional_hc3')
nonparam=bh(pd.DataFrame(nonparam_rows),'mw_p','q_bh_conditional_mw')
nonparam=bh(nonparam,'bf_p','q_bh_conditional_brown_forsythe')
conditional.to_csv(TAB/'conditioned_model_effects.csv',index=False)
nonparam.to_csv(TAB/'conditioned_nonparametric_effects.csv',index=False)
assert len(conditional)==len(nonparam)
print(f'Adjusted marginal contrasts: {len(marginal)}; pair-condition contrasts: {len(conditional)}')
display(conditional.groupby('status').size().rename('n').to_frame())

Adjusted marginal contrasts: 56; pair-condition contrasts: 406


,n
status,
ok,406


## 4 — Decomposing three- and four-way interactions

Three-way *difference-in-differences-in-differences* contrasts from M3 are computed at prespecified extreme levels (the first and last analysis categories for each factor) and standardized across the observed distribution of the fourth factor. For M4, the fourth-order contrast requires 16 combinations. It is withheld if the saturated fit has unreliable robust covariance, or any required corner has fewer than five observations. The table retains the exact reason for non-estimation.

In [5]:
def signed_corner_contrast(col,stage,axes,base_distribution):
    base=base_distribution[ANA_KEYS].copy()
    design=DESIGNS[stage]
    v=np.zeros(len(design['columns']),dtype=float)
    for bits in product([0,1],repeat=len(axes)):
        x=base.copy()
        for factor,bit in zip(axes,bits):
            x[ANA[factor]]=LEVELS[factor][-1 if bit else 0]
        for factor in FACTOR_KEYS:
            x[ANA[factor]]=pd.Categorical(x[ANA[factor]],categories=LEVELS[factor])
        sign=(-1)**(len(axes)-sum(bits))
        v+=sign*matrix_for(stage,x).mean(axis=0)
    fit=FITS[col,stage]
    est=float(v@fit.params)
    cov=np.asarray(fit.cov_params())
    variance=float(v@cov@v)
    se=np.sqrt(max(0,variance)) if np.isfinite(variance) else np.nan
    p=float(2*st.norm.sf(abs(est/se))) if se>0 and np.isfinite(se) else np.nan
    return est,float(est-1.9599639845*se),float(est+1.9599639845*se),p

three_rows=[];four_rows=[]
for col in HRV:
    stable3=bool(hierarchy.loc[(hierarchy.hrv_variable==col)&(hierarchy.model_stage=='M3'),'robust_inference_ok'].iloc[0])
    stable4=bool(hierarchy.loc[(hierarchy.hrv_variable==col)&(hierarchy.model_stage=='M4'),'robust_inference_ok'].iloc[0])
    for triple in combinations(FACTOR_KEYS,3):
        remaining=next(f for f in FACTOR_KEYS if f not in triple)
        cells=[]
        for bits in product([0,1],repeat=3):
            subset=df
            for factor,bit in zip(triple,bits):
                subset=subset.loc[subset[ANA[factor]]==LEVELS[factor][-1 if bit else 0]]
            cells.append(len(subset))
        rec={'hrv_variable':col,'triplet':'_x_'.join(SHORT[f] for f in triple),
             'standardized_over_factor':remaining,'minimum_three_corner_n':min(cells),
             'model_stage':'M3','outcome_scale':SCALES[col],
             'status':'ok' if stable3 and min(cells)>=MIN_CELL_N else 'low_cell_support' if min(cells)<MIN_CELL_N else 'unstable_model'}
        if stable3 and min(cells)>=MIN_CELL_N:
            est,lo,hi,p=signed_corner_contrast(col,'M3',triple,df)
            rec.update(estimate=est,ci95_lower=lo,ci95_upper=hi,p_hc3=p)
        three_rows.append(rec)
    corners=[]
    for bits in product([0,1],repeat=4):
        subset=df
        for factor,bit in zip(FACTOR_KEYS,bits):
            subset=subset.loc[subset[ANA[factor]]==LEVELS[factor][-1 if bit else 0]]
        corners.append(len(subset))
    rec={'hrv_variable':col,'model_stage':'M4','outcome_scale':SCALES[col],
        'contrast':'four-way difference-in-differences of differences',
        'minimum_16_corner_n':min(corners),'n_corners_below_5':int(sum(n<MIN_CELL_N for n in corners)),
        'hc3_reliable':stable4,'status':'ok' if stable4 and min(corners)>=MIN_CELL_N else
            'insufficient_cell_support' if min(corners)<MIN_CELL_N else 'unstable_hc3'}
    if rec['status']=='ok':
        est,lo,hi,p=signed_corner_contrast(col,'M4',FACTOR_KEYS,df)
        rec.update(estimate=est,ci95_lower=lo,ci95_upper=hi,p_hc3=p)
    four_rows.append(rec)
three_decomp=bh(pd.DataFrame(three_rows),'p_hc3','q_bh_3way_decomposition')
four_decomp=bh(pd.DataFrame(four_rows),'p_hc3','q_bh_4way_decomposition')
three_decomp.to_csv(TAB/'three_way_interaction_decomposition.csv',index=False)
four_decomp.to_csv(TAB/'four_way_interaction_feasibility.csv',index=False)
print('Three-way decompositions: ',three_decomp.status.value_counts().to_dict())
print('Four-way decomposition: ',four_decomp.status.value_counts().to_dict())

Three-way decompositions:  {'ok': 28}
Four-way decomposition:  {'insufficient_cell_support': 7}


## 5 — Four-factor models with continuous physiological measurements

Both specifications retain all four substantive factors, with continuous age, BMI and either SBP or DBP, and categorical sex. All continuous variables are centered and measured in human-readable increments (10 years, 5 BMI units, 10 mmHg). The two blood pressure specifications are **alternative measurements of the same factor**, not five-factor models. M1–M4 are fit independently for each HRV measurement. Model coefficients are on the documented HRV scale. Confidence intervals and Wald tests are marked unavailable when robust covariance is unstable.

In [6]:
CONTINUOUS_SPECS={
    'age_sex_sbp_bmi':('bp_systolic','Systolic BP'),
    'age_sex_dbp_bmi':('bp_diastolic','Diastolic BP'),
}
df['age_10']=(df['age']-df['age'].mean())/10
for raw,new,step in [('imc','bmi_5',5),('bp_systolic','sbp_10',10),('bp_diastolic','dbp_10',10)]:
    df[new]=(df[raw]-df[raw].mean())/step

CONT_MODELS={};CONT_DESIGNS={};cont_rows=[];cont_terms=[]
for si,(spec,(bp_raw,bp_label)) in enumerate(CONTINUOUS_SPECS.items()):
    base=['age_10','C(sex_category, Sum)','sbp_10' if bp_raw=='bp_systolic' else 'dbp_10','bmi_5']
    for stage,order in STAGES.items():
        blocks={k:[':'.join(combo) for combo in combinations(base,k)] for k in range(1,5)}
        terms=[term for k in range(1,order+1) for term in blocks[k]]
        x=dmatrix(' + '.join(terms),df,return_type='dataframe')
        CONT_DESIGNS[(spec,stage)]={'matrix':x,'info':x.design_info,'block':blocks[order],
                                   'columns':list(x.columns)}
    for oi,col in enumerate(HRV):
        y=outcome(col)
        prev_rss=None
        for stage,order in STAGES.items():
            design=CONT_DESIGNS[spec,stage]
            X=design['matrix'].to_numpy(float)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore',RuntimeWarning)
                model,diag=fit_robust(y,X)
            CONT_MODELS[(col,spec,stage)]=model
            rss=float(np.sum(model.resid**2))
            rec={'hrv_variable':col,'specification':spec,'bp_definition':bp_label,'stage':stage,
                 'outcome_scale':SCALES[col],'n':len(y),'r2':float(model.rsquared),
                 'aic':float(model.aic),'bic':float(model.bic),'rss':rss,
                 'partial_r2_added_block':max(0,(prev_rss-rss)/prev_rss) if prev_rss is not None and prev_rss>0 else np.nan,
                 **diag}
            rec['block_p_hc3']=wald_block(model,design['info'],design['block']) if diag['robust_inference_ok'] else np.nan
            rec['status']='ok' if diag['robust_inference_ok'] else 'unstable_hc3'
            rec.update(cv_model(y,X,SEED+si*20+oi))
            cont_rows.append(rec)
            if diag['robust_inference_ok']:
                for term in design['block']:
                    cont_terms.append({'hrv_variable':col,'specification':spec,'stage':stage,
                        'term':term,'interaction_order':order,'p_hc3':wald_term(model,design['info'],term)})
            prev_rss=rss
continuous=bh(pd.DataFrame(cont_rows),'block_p_hc3','block_q_bh_all_continuous')
cont_term_table=bh(pd.DataFrame(cont_terms),'p_hc3','term_q_bh_all_continuous')
continuous.to_csv(TAB/'continuous_four_factor_models.csv',index=False)
cont_term_table.to_csv(TAB/'continuous_interaction_terms.csv',index=False)
assert len(continuous)==len(HRV)*len(CONTINUOUS_SPECS)*4
print('Continuous/mixed model status:')
display(continuous.groupby(['stage','status']).size().rename('n').to_frame())

Continuous/mixed model status:


,,n
stage,status,
M1,ok,14
M2,ok,14
M3,ok,14
M4,ok,14


## 6 — Quantile heterogeneity and nonlinear sensitivity

Adjusted quantile regression is exploratory and uses an additive four-factor specification (age, sex, continuous BP, BMI) to check whether conditional lower, median and upper HRV quantiles behave differently. The spline sensitivity compares a prespecified linear M1 continuous model with one nonlinear continuous term at a time; the difference in in-sample R² is descriptive, while five-fold CV RMSE is reported separately. These analyses do **not** infer nonlinear four-way interactions.

In [7]:
quantile_rows=[]
nonlinear_rows=[]
for si,(spec,(bp_raw,_)) in enumerate(CONTINUOUS_SPECS.items()):
    bpterm='sbp_10' if bp_raw=='bp_systolic' else 'dbp_10'
    additive=['age_10','C(sex_category, Sum)',bpterm,'bmi_5']
    Xbase=CONT_DESIGNS[spec,'M1']['matrix']
    Xquant=Xbase.to_numpy(float)
    for hi,col in enumerate(HRV):
        y=outcome(col)
        for q in (.25,.50,.75):
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    qr=sm.QuantReg(y,Xquant).fit(q=q,max_iter=2200,p_tol=1e-6)
                for name,est,se,p in zip(Xbase.columns,qr.params,qr.bse,qr.pvalues):
                    quantile_rows.append({'hrv_variable':col,'specification':spec,'quantile':q,
                        'term':name,'estimate':float(est),'standard_error':float(se),
                        'p_nominal':float(p),'outcome_scale':SCALES[col],'status':'ok'})
            except (ValueError,np.linalg.LinAlgError,ZeroDivisionError):
                quantile_rows.append({'hrv_variable':col,'specification':spec,'quantile':q,
                    'term':'model','status':'not_estimable','outcome_scale':SCALES[col]})
        baseline=continuous.loc[(continuous.hrv_variable==col)&(continuous.specification==spec)&(continuous.stage=='M1')].iloc[0]
        for spline_term in ('age_10',bpterm,'bmi_5'):
            rhs=' + '.join('bs('+a+', df=4, degree=3, include_intercept=False)' if a==spline_term else a for a in additive)
            X=dmatrix(rhs,df,return_type='dataframe').to_numpy(float)
            try:
                if np.linalg.matrix_rank(X)<X.shape[1]:
                    raise ValueError('rank deficient')
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore',RuntimeWarning)
                    model,diag=fit_robust(y,X)
                cv=cv_model(y,X,SEED+si*100+hi)
                nonlinear_rows.append({'hrv_variable':col,'specification':spec,'spline_factor':spline_term,
                    'baseline_r2':baseline.r2,'spline_r2':float(model.rsquared),
                    'delta_r2':float(model.rsquared-baseline.r2),
                    'baseline_cv_rmse':baseline.cv_rmse_model_scale,
                    'spline_cv_rmse':cv['cv_rmse_model_scale'],
                    'delta_cv_rmse':cv['cv_rmse_model_scale']-baseline.cv_rmse_model_scale,
                    'cv_valid_folds':cv['cv_valid_folds'],
                    'max_leverage':diag['max_leverage'],
                    'status':'ok' if diag['robust_inference_ok'] else 'unstable_hc3'})
            except (ValueError,np.linalg.LinAlgError):
                nonlinear_rows.append({'hrv_variable':col,'specification':spec,'spline_factor':spline_term,
                    'status':'not_estimable'})
quantiles=pd.DataFrame(quantile_rows)
nonlinear=pd.DataFrame(nonlinear_rows)
quantiles.to_csv(TAB/'adjusted_quantile_regression.csv',index=False)
nonlinear.to_csv(TAB/'nonlinear_additive_sensitivity.csv',index=False)
print(f'Quantile regression coefficients: {len(quantiles)}; nonlinear comparisons: {len(nonlinear)}')

Quantile regression coefficients: 210; nonlinear comparisons: 42


## 7 — Model sensitivity, bootstrap stability and influence diagnostics

The core M2/M3 interaction findings are re-estimated after (a) excluding the outer 1% of the HRV distribution for each outcome and (b) switching between identity and log10 outcome scales when valid. A third specification merges blood pressure into a deliberately coarser two-level definition. These are explicitly **sensitivity analyses**, not arbitrary recoding to obtain significant results. Nonparametric bootstrap evaluates stability of the added-block partial R². Reproducible seeds are fixed; bootstrap repetitions are user-configurable.

In [8]:
sensitivity_rows=[]
for hi,col in enumerate(HRV):
    lo,high=df[col].quantile([.01,.99])
    base=SCALES[col]
    alternative='identity' if base=='log10' else 'log10'
    for variant,sub,scale in [('trim_outer_1pct_each_tail',df.loc[df[col].between(lo,high)],base),
                              ('alternative_scale',df,alternative)]:
        if scale=='log10' and (sub[col]<=0).any():
            for stage in ('M2','M3'):
                sensitivity_rows.append({'hrv_variable':col,'variant':variant,'stage':stage,
                    'status':'not_estimable_nonpositive_values','n':len(sub)})
            continue
        y=outcome(col,scale,sub)
        prev_rss=None
        for stage in ('M1','M2','M3'):
            X=build_design_matrices([DESIGNS[stage]['info']],sub,return_type='dataframe')[0]
            with warnings.catch_warnings():
                warnings.simplefilter('ignore',RuntimeWarning)
                m,diag=fit_robust(y,X)
            rss=float(np.dot(m.resid,m.resid))
            if stage in ('M2','M3'):
                sensitivity_rows.append({'hrv_variable':col,'variant':variant,'stage':stage,
                    'status':'ok' if diag['robust_inference_ok'] else 'unstable_hc3',
                    'n':len(sub),'scale':scale,'r2':float(m.rsquared),
                    'block_partial_r2':max(0.,(prev_rss-rss)/prev_rss) if prev_rss>0 else np.nan,
                    'block_p_hc3':wald_block(m,DESIGNS[stage]['info'],DESIGNS[stage]['block_terms']) if diag['robust_inference_ok'] else np.nan,
                    'max_leverage':diag['max_leverage']})
            prev_rss=rss
    # Predeclared two-level BP sensitivity: lower (<Stage 1) versus elevated (Stage 1–3).
    d=df.copy()
    d['bp_binary']=pd.Categorical(np.where(d['bp_category'].isin(['Stage 1','Stage 2','Stage 3']),
                'Stage 1-3','Below Stage 1'),categories=['Below Stage 1','Stage 1-3'])
    spec_terms={**TERMS,'bp_category':'C(bp_binary, Sum)'}
    y=outcome(col)
    prev_rss=None
    for order in range(1,4):
        rhs=' + '.join(':'.join(spec_terms[f] for f in factors)
            for k in range(1,order+1) for factors in combinations(FACTOR_KEYS,k))
        X=dmatrix(rhs,d,return_type='dataframe')
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',RuntimeWarning)
            m,diag=fit_robust(y,X)
        rss=float(np.dot(m.resid,m.resid))
        if order>1:
            sensitivity_rows.append({'hrv_variable':col,'variant':'binary_bp_grouping',
                'stage':f'M{order}','status':'ok' if diag['robust_inference_ok'] else 'unstable_hc3',
                'n':len(d),'scale':base,'r2':float(m.rsquared),
                'block_partial_r2':max(0.,(prev_rss-rss)/prev_rss) if prev_rss>0 else np.nan,
                'max_leverage':diag['max_leverage']})
        prev_rss=rss
sensitivity=pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(TAB/'four_factor_sensitivity.csv',index=False)

# Bootstrap partial-R² gains for the two inferentially useful interaction blocks.
stability_rows=[]
Xm={stage:DESIGNS[stage]['matrix'].to_numpy(float) for stage in ('M1','M2','M3')}
rng=np.random.default_rng(SEED+717)
boot_indices=[rng.integers(0,len(df),size=len(df)) for _ in range(N_BOOT)]
for hi,col in enumerate(HRV):
    y=outcome(col)
    observed=hierarchy.loc[hierarchy.hrv_variable==col].set_index('model_stage')
    effects={'M2':[],'M3':[]}
    ranks={'M2':0,'M3':0}
    for idx in boot_indices:
        rss={}
        for stage in ('M1','M2','M3'):
            X=Xm[stage][idx]
            if np.linalg.matrix_rank(X)<X.shape[1]:
                break
            beta=np.linalg.lstsq(X,y[idx],rcond=None)[0]
            res=y[idx]-X@beta
            rss[stage]=float(res@res)
        if 'M2' in rss and rss['M1']>0:
            effects['M2'].append(max(0.,(rss['M1']-rss['M2'])/rss['M1']))
            ranks['M2']+=1
        if 'M3' in rss and rss['M2']>0:
            effects['M3'].append(max(0.,(rss['M2']-rss['M3'])/rss['M2']))
            ranks['M3']+=1
    for stage in ('M2','M3'):
        arr=np.asarray(effects[stage])
        stability_rows.append({'hrv_variable':col,'stage':stage,'observed_incremental_partial_r2':observed.loc[stage,'incremental_partial_r2'],
            'bootstrap_requested':N_BOOT,'bootstrap_rank_valid':ranks[stage],
            'bootstrap_valid_fraction':ranks[stage]/N_BOOT,
            'bootstrap_median':float(np.median(arr)) if len(arr) else np.nan,
            'bootstrap_ci95_lower':float(np.quantile(arr,.025)) if len(arr) else np.nan,
            'bootstrap_ci95_upper':float(np.quantile(arr,.975)) if len(arr) else np.nan})
stability=pd.DataFrame(stability_rows)
stability.to_csv(TAB/'interaction_bootstrap_stability.csv',index=False)

influence_rows=[]
for col in HRV:
    for stage in ('M1','M2','M3','M4'):
        fit=FITS[col,stage]
        diag=fit.get_influence()
        h=np.asarray(diag.hat_matrix_diag,float)
        cook=np.asarray(diag.cooks_distance[0],float)
        influence_rows.append({'hrv_variable':col,'stage':stage,'n':len(h),'max_leverage':float(np.max(h)),
            'n_leverage_ge_095':int((h>=.95).sum()),
            'max_cooks_d':float(np.nanmax(cook)),
            'n_cooks_d_above_4_n':int(np.sum(cook>4/len(df))),
            'robust_inference_ok':bool(hierarchy.loc[(hierarchy.hrv_variable==col)&(hierarchy.model_stage==stage),'robust_inference_ok'].iloc[0])})
influence=pd.DataFrame(influence_rows)
influence.to_csv(TAB/'model_influence_diagnostics.csv',index=False)
print('Sensitivity records:',len(sensitivity),'| Bootstrap stability records:',len(stability))
display(stability.round(3))

Sensitivity records: 42 | Bootstrap stability records: 14


/home/david/miniconda3/envs/hrv/lib/python3.14/site-packages/statsmodels/stats/outliers_influence.py:961: RuntimeWarning: invalid value encountered in sqrt
  return self.resid / sigma / np.sqrt(1 - hii)
/home/david/miniconda3/envs/hrv/lib/python3.14/site-packages/statsmodels/stats/outliers_influence.py:961: RuntimeWarning: invalid value encountered in sqrt
  return self.resid / sigma / np.sqrt(1 - hii)
/home/david/miniconda3/envs/hrv/lib/python3.14/site-packages/statsmodels/stats/outliers_influence.py:961: RuntimeWarning: invalid value encountered in sqrt
  return self.resid / sigma / np.sqrt(1 - hii)
/home/david/miniconda3/envs/hrv/lib/python3.14/site-packages/statsmodels/stats/outliers_influence.py:961: RuntimeWarning: invalid value encountered in sqrt
  return self.resid / sigma / np.sqrt(1 - hii)
/home/david/miniconda3/envs/hrv/lib/python3.14/site-packages/statsmodels/stats/outliers_influence.py:961: RuntimeWarning: invalid value encountered in sqrt
  return self.resid / sigma / np

,hrv_variable,stage,observed_incremental_partial_r2,bootstrap_requested,bootstrap_rank_valid,bootstrap_valid_fraction,bootstrap_median,bootstrap_ci95_lower,bootstrap_ci95_upper
0,t2m_pre_mean_rr,M2,0.031,150,150,1.0,0.059,0.027,0.101
1,t2m_pre_mean_rr,M3,0.021,150,150,1.0,0.044,0.018,0.081
2,t2m_pre_mean_hr,M2,0.031,150,150,1.0,0.057,0.029,0.095
3,t2m_pre_mean_hr,M3,0.021,150,150,1.0,0.042,0.018,0.079
4,t2m_pre_sdnn,M2,0.052,150,150,1.0,0.080,0.035,0.130
5,t2m_pre_sdnn,M3,0.020,150,150,1.0,0.047,0.018,0.088
6,t2m_pre_rmssd,M2,0.028,150,150,1.0,0.051,0.027,0.090
7,t2m_pre_rmssd,M3,0.010,150,150,1.0,0.031,0.013,0.063
8,t2m_pre_hf,M2,0.012,150,150,1.0,0.037,0.016,0.073
9,t2m_pre_hf,M3,0.031,150,150,1.0,0.052,0.024,0.096


## 8 — Integrated synthesis and optional cross-notebook evidence

The M1→M4 analysis produces a self-contained model progression. If the CSVs from Notebooks 07 and 08 are present in the sibling output directories, the notebook also imports their reported interaction summaries; absence of those optional files never prevents execution. P-values from different model families are **not pooled into a single FDR correction** and the different model specifications are not assumed to be directly interchangeable.

In [9]:
integrated=[]
for col,(slug,label,unit) in HRV.items():
    rec={'hrv_variable':col,'hrv_label':label,'outcome_scale':SCALES[col]}
    sub=hierarchy[hierarchy.hrv_variable==col].set_index('model_stage')
    for stage in STAGES:
        r=sub.loc[stage]
        rec[f'{stage}_r2']=r.r2
        rec[f'{stage}_partial_r2']=r.incremental_partial_r2
        rec[f'{stage}_block_p_hc3']=r.added_block_p_hc3
        rec[f'{stage}_block_q_bh']=r.added_block_q_bh
        rec[f'{stage}_cv_rmse']=r.cv_rmse_model_scale
        rec[f'{stage}_cv_valid_folds']=r.cv_valid_folds
        rec[f'{stage}_status']=r.model_status
    for spec in CONTINUOUS_SPECS:
        s=continuous[(continuous.hrv_variable==col)&(continuous.specification==spec)].set_index('stage')
        rec[f'{spec}_M4_status']=s.loc['M4','status']
        rec[f'{spec}_M4_q']=s.loc['M4','block_q_bh_all_continuous']
    rec['most_supported_categorical_stage']=next((stage for stage in reversed(list(STAGES))
        if sub.loc[stage,'robust_inference_ok'] and sub.loc[stage,'cv_valid_folds']==CV_FOLDS),'none')
    integrated.append(rec)
synthesis=pd.DataFrame(integrated)
synthesis.to_csv(TAB/'four_factor_evidence_synthesis.csv',index=False)

previous_rows=[]
known_sources={
    '07_bifactorial':('07_bifactorial_hrv_analysis','categorical_factorial_models.csv','pair','interaction_q_bh_42'),
    '08_trifactorial':('08_trifactorial_hrv_analysis','categorical_three_way_models.csv','triplet','three_way_q_bh_28'),
}
for name,(folder,filename,key,q_col) in known_sources.items():
    file=OUT.parent/folder/'tables'/filename
    if file.exists():
        old=pd.read_csv(file)
        for row in old.itertuples(index=False):
            d=row._asdict()
            previous_rows.append({'source_notebook':name,'source_available':True,'hrv_variable':d['hrv_variable'],
                'factor_combination':d[key],'source_adjusted_p':d.get(q_col,np.nan),
                'note':'Source model and correction family differ from Notebook 09; exploratory comparison only.'})
    else:
        previous_rows.append({'source_notebook':name,'source_available':False,'hrv_variable':'',
            'factor_combination':'','source_adjusted_p':np.nan,
            'note':'Optional previous-stage output not found; this is not a failure of Notebook 09.'})
previous=pd.DataFrame(previous_rows)
previous.to_csv(TAB/'previous_stage_interaction_reference.csv',index=False)
print('Saved integrated synthesis for all 7 HRV measures and optional predecessor references.')
display(synthesis[['hrv_label','M1_r2','M2_r2','M3_r2','M4_r2','M4_status','most_supported_categorical_stage']].round(3))

Saved integrated synthesis for all 7 HRV measures and optional predecessor references.


,hrv_label,M1_r2,M2_r2,M3_r2,M4_r2,M4_status,most_supported_categorical_stage
0,Mean RR,0.022,0.053,0.073,0.074,unstable_hc3,M3
1,Mean HR,0.019,0.050,0.069,0.070,unstable_hc3,M3
2,SDNN,0.113,0.159,0.175,0.176,unstable_hc3,M3
3,RMSSD,0.013,0.040,0.050,0.054,unstable_hc3,M3
4,HF,0.026,0.038,0.067,0.073,unstable_hc3,M3
5,LF,0.031,0.065,0.091,0.099,unstable_hc3,M3
6,VLF,0.036,0.062,0.091,0.103,unstable_hc3,M3


## 9 — Figures by outcome, not by the entire dataset

All HRV-specific figure families are nested under their individual HRV directory. Colors follow the factor being displayed; facet rows and columns encode the other factors. Original and analysis-only category figures are explicitly distinguished. Effects and tests shown on transformed-model scales are labeled accordingly, and unsupported fourth-order inference is never marked significant.

In [10]:
def log_display(col):
    s=df[col]
    return bool(s.min()>0 and s.quantile(.99)/max(s.quantile(.01),1e-12)>30)

# Original-category distribution panels and analytical-category median maps.
for col,(slug,label,unit) in HRV.items():
    root=FIG/slug/'01_descriptive'
    ages=FACTORS['age_group'];sexes=FACTORS['sex_category']
    fig,axes=plt.subplots(3,2,figsize=(19,14.5),squeeze=False,sharey=False)
    for i,age in enumerate(ages):
        for j,sex in enumerate(sexes):
            ax=axes[i,j]
            subset=df[(df.age_group==age)&(df.sex_category==sex)]
            sns.boxplot(data=subset,x='bp_category',y=col,hue='bmi_category',
                order=FACTORS['bp_category'],hue_order=FACTORS['bmi_category'],
                palette=dict(zip(FACTORS['bmi_category'],sns.color_palette('flare',4))),
                linewidth=1,showfliers=False,ax=ax)
            if log_display(col):ax.set_yscale('log')
            ax.set(title=f'Age {age} | {sex} | n={len(subset)}',xlabel='Blood pressure',ylabel=f'{label} ({unit})')
            ax.tick_params(axis='x',rotation=32,labelsize=8)
            if i!=0 or j!=1:
                legend=ax.get_legend()
                if legend:legend.remove()
            else:
                ax.legend(title='BMI',fontsize=9,title_fontsize=9,loc='best')
    fig.suptitle(f'{label} | all original factor categories',fontsize=19,y=1.020)
    fig.text(.5,.985,'Descriptive only; small combinations are not used to assert differences.',ha='center',fontsize=11,color='#56647a')
    fig.tight_layout(rect=(0,0,1,.948),h_pad=3.3,w_pad=1.8)
    fig.savefig(root/'01_original_four_factor_boxplots.png',dpi=170,bbox_inches='tight')
    plt.close(fig)

    ages=LEVELS['age_group']
    fig,axes=plt.subplots(2,2,figsize=(13.5,10.8),squeeze=False)
    v=descriptive.loc[(descriptive.hrv_variable==col)&(descriptive.scheme=='analysis')]
    lo=float(v['median'].quantile(.05));hi=float(v['median'].quantile(.95))
    for i,age in enumerate(ages):
        for j,sex in enumerate(sexes):
            ax=axes[i,j]
            sub=v[(v.age_level==age)&(v.sex_level==sex)]
            med=sub.pivot(index='bp_level',columns='bmi_level',values='median').reindex(
                index=LEVELS['bp_category'],columns=LEVELS['bmi_category'])
            counts=sub.pivot(index='bp_level',columns='bmi_level',values='n').reindex(
                index=LEVELS['bp_category'],columns=LEVELS['bmi_category'])
            annot=np.array([[f'{med.iloc[r,c]:.1f}\nn={counts.iloc[r,c]:.0f}'
                    for c in range(3)] for r in range(3)])
            sns.heatmap(med,ax=ax,annot=annot,fmt='',cmap='YlGnBu',vmin=lo,vmax=hi,
                linewidths=.8,linecolor='white',annot_kws={'fontsize':9},
                cbar=(i==0 and j==1),cbar_kws={'label':f'Median {label} ({unit})'})
            ax.set(title=f'Age {age} | {sex}',xlabel='BMI',ylabel='Blood pressure')
            ax.tick_params(axis='x',rotation=25)
    fig.suptitle(f'{label} | analysis-group medians and cell sizes',y=.998,fontsize=18)
    fig.tight_layout(rect=(0,0,1,.964),h_pad=3)
    fig.savefig(root/'02_analysis_cell_medians.png',bbox_inches='tight',dpi=170)
    plt.close(fig)

    rows=hierarchy[hierarchy.hrv_variable==col].set_index('model_stage').reindex(list(STAGES))
    fig,axes=plt.subplots(1,2,figsize=(13,4.9))
    sns.lineplot(x=list(STAGES),y=rows.r2.to_numpy(),marker='o',lw=2.5,ax=axes[0],color='#2c647c')
    for idx,(stage,row) in enumerate(rows.iterrows()):
        axes[0].annotate(f'{row.r2:.3f}',(idx,row.r2),xytext=(2,7),textcoords='offset points',fontsize=9)
    axes[0].set(title='In-sample model fit',xlabel='Model hierarchy',ylabel='R²')
    sns.lineplot(x=list(STAGES),y=rows.cv_rmse_model_scale.to_numpy(),marker='o',lw=2.5,
        ax=axes[1],color='#99632d')
    for idx,(stage,row) in enumerate(rows.iterrows()):
        axes[1].annotate(f'{int(row.cv_valid_folds)}/{CV_FOLDS} valid folds',(idx,row.cv_rmse_model_scale),
            xytext=(0,9),textcoords='offset points',fontsize=8,ha='center')
    axes[1].set(title='Five-fold out-of-sample error',xlabel='Model hierarchy',ylabel=f'CV RMSE ({SCALES[col]} outcome scale)')
    for idx,(stage,row) in enumerate(rows.iterrows()):
        if not row.robust_inference_ok:
            axes[0].axvspan(idx-.3,idx+.3,color='#b5654a',alpha=.08)
    fig.suptitle(f'{label} | four-factor model progression',fontsize=18)
    fig.tight_layout(rect=(0,0,1,.94))
    fig.savefig(FIG/slug/'02_model_hierarchy'/'01_fit_and_cv.png',bbox_inches='tight',dpi=170)
    plt.close(fig)

    subset=marginal[marginal.hrv_variable==col].copy()
    subset['comparison']=subset.factor.map(SHORT)+': '+subset.level_1+' vs '+subset.level_2
    subset=subset.sort_values(['factor','level_1','level_2']).reset_index(drop=True)
    fig,ax=plt.subplots(figsize=(13.8,max(5.1,.62*len(subset)+2)))
    for i,row in subset.iterrows():
        ax.plot([row.ci95_lower,row.ci95_upper],[i,i],color='#567489',lw=2)
        ax.scatter(row.estimate,i,color='#176b89',s=52,zorder=3)
    ax.set_yticks(range(len(subset)),subset.comparison)
    ax.invert_yaxis();ax.axvline(0,color='#343b47',lw=1,ls='--')
    ax.set(xlabel=f'Standardized difference ({SCALES[col]} model scale)',ylabel='',
           title=f'{label} | adjusted marginal effects, M1 HC3')
    fig.tight_layout()
    fig.savefig(FIG/slug/'04_adjusted_effects'/'01_m1_marginal_effects.png',dpi=170,bbox_inches='tight')
    plt.close(fig)

    for pair in combinations(FACTOR_KEYS,2):
        name='_x_'.join(SHORT[f] for f in pair)
        sub=conditional[(conditional.hrv_variable==col)&(conditional.pair==name)].copy()
        sub['stratum']=sub.conditioning_factor.map(SHORT)+': '+sub.conditioning_level
        sub['contrast']=sub.focal_factor.map(SHORT)+': '+sub.focal_level_1+' vs '+sub.focal_level_2
        mat=sub.pivot(index='stratum',columns='contrast',values='estimate')
        # A grey hatch is represented by missing entries. Unsupported tests are never imputed.
        fig,ax=plt.subplots(figsize=(max(10.5,2.6*len(mat.columns)+2),max(4.8,.55*len(mat)+2.3)))
        if mat.notna().any().any():
            bound=max(.01,float(np.nanquantile(np.abs(mat.to_numpy()),.95)))
            sns.heatmap(mat,ax=ax,cmap='vlag',center=0,vmin=-bound,vmax=bound,
                annot=True,fmt='.2g',linewidths=.5,linecolor='white',
                cbar_kws={'label':f'HC3 contrast ({SCALES[col]})'})
            ax.tick_params(axis='x',rotation=35)
            for tick in ax.get_xticklabels():tick.set_ha('right')
        else:
            ax.text(.5,.5,'No eligible comparisons',ha='center',va='center',transform=ax.transAxes)
        ax.set(title=f'{label} | conditional contrasts | {name}',xlabel='Focal-level comparison',ylabel='Conditioning stratum')
        fig.tight_layout()
        fig.savefig(FIG/slug/'03_pair_interactions'/f'{name}__conditional_effects.png',dpi=165,bbox_inches='tight')
        plt.close(fig)

    sub=three_decomp[three_decomp.hrv_variable==col]
    fig,ax=plt.subplots(figsize=(12,5.2))
    positions=np.arange(len(sub))
    for k,row in enumerate(sub.itertuples()):
        if row.status=='ok':
            ax.plot([row.ci95_lower,row.ci95_upper],[k,k],lw=2.5,color='#277b85')
            ax.scatter(row.estimate,k,color='#1a6476',s=60)
        else:
            ax.text(.03,k,'not estimable: '+row.status,color='#a04343',fontsize=9,va='center')
    ax.set_yticks(positions,sub.triplet)
    ax.set_xlim(left=min(-.5,ax.get_xlim()[0]))
    ax.axvline(0,color='#475569',ls='--',lw=1)
    ax.invert_yaxis()
    ax.set(title=f'{label} | triple interaction decomposition (M3)',
           xlabel=f'Triple difference-in-differences ({SCALES[col]} model scale)',ylabel='')
    fig.tight_layout()
    fig.savefig(FIG/slug/'05_higher_interactions'/'01_three_way_contrasts.png',dpi=170,bbox_inches='tight')
    plt.close(fig)

    # Outcome-specific diagnostic plots for M3 (the most complex generally supportable categorical model).
    model=FITS[col,'M3']
    pred=model.fittedvalues
    resid=model.resid
    fig,axes=plt.subplots(1,2,figsize=(12.5,5))
    sns.scatterplot(x=pred,y=resid,ax=axes[0],s=26,alpha=.52,color='#247993',edgecolor=None)
    axes[0].axhline(0,ls='--',color='#3a3f48')
    axes[0].set(title='Fitted vs residuals',xlabel='Fitted value',ylabel='Residual')
    st.probplot(resid,dist='norm',plot=axes[1])
    axes[1].set_title('Normal Q–Q of residuals (descriptive diagnostic)')
    fig.suptitle(f'{label} | M3 model diagnostics',fontsize=17)
    fig.tight_layout(rect=(0,0,1,.945))
    fig.savefig(FIG/slug/'07_diagnostics'/'01_residual_diagnostics.png',bbox_inches='tight',dpi=170)
    plt.close(fig)

    # Three quantile regression estimates of continuous effects (SBP specification).
    q=quantiles[(quantiles.hrv_variable==col)&(quantiles.specification=='age_sex_sbp_bmi')&
        (quantiles.term.isin(['age_10','sbp_10','bmi_5']))&(quantiles.status=='ok')]
    fig,ax=plt.subplots(figsize=(9.6,5.3))
    if not q.empty:
        sns.lineplot(data=q,x='quantile',y='estimate',hue='term',style='term',markers=True,
                     dashes=False,lw=2.2,palette='colorblind',ax=ax)
    ax.axhline(0,ls='--',color='#526176')
    ax.set(xlabel='Conditional HRV quantile',ylabel=f'Coefficient ({SCALES[col]} scale)',
           title=f'{label} | additive quantile heterogeneity (SBP specification)')
    fig.tight_layout()
    fig.savefig(FIG/slug/'07_diagnostics'/'02_quantile_heterogeneity.png',bbox_inches='tight',dpi=170)
    plt.close(fig)
print('Generated descriptive, marginal, pair-conditional, hierarchy, triplet, and diagnostic plots by HRV metric.')

Generated descriptive, marginal, pair-conditional, hierarchy, triplet, and diagnostic plots by HRV metric.


## 10 — Continuous-model interaction figures and global evidence maps

These plots depict the predicted HRV response as age varies, stratified by sex and faceted across three blood-pressure settings, with BMI fixed at the cohort median. They use the **most complex robustly supported specification** for each outcome and blood-pressure measurement. They are adjusted model predictions, not longitudinal individual trajectories.

In [11]:
for col,(slug,label,unit) in HRV.items():
    for spec,(bpraw,bplabel) in CONTINUOUS_SPECS.items():
        r=continuous[(continuous.hrv_variable==col)&(continuous.specification==spec)]
        possible=r[(r.robust_inference_ok)&(r.cv_valid_folds==CV_FOLDS)]
        if possible.empty:
            continue
        stage=max(possible.stage,key=lambda s:STAGES[s])
        design=CONT_DESIGNS[spec,stage]
        fit=CONT_MODELS[(col,spec,stage)]
        bps=np.quantile(df[bpraw],[.1,.5,.9])
        agevalues=np.linspace(df.age.min(),df.age.max(),75)
        fig,axes=plt.subplots(1,3,figsize=(16.8,5.4),sharey=True)
        for ax,bp_value,q in zip(axes,bps,(.1,.5,.9)):
            for sex in FACTORS['sex_category']:
                d=pd.DataFrame({'age_10':(agevalues-df.age.mean())/10,
                    'bmi_5':(np.median(df.imc)-df.imc.mean())/5,
                    'sbp_10':(bp_value-df.bp_systolic.mean())/10,
                    'dbp_10':(bp_value-df.bp_diastolic.mean())/10,
                    'sex_category':sex})
                X=build_design_matrices([design['info']],d,return_type='dataframe')[0].to_numpy(float)
                pred=X@fit.params
                covariance=np.asarray(fit.cov_params(),float)
                errs=np.sqrt(np.maximum(0,np.einsum('ij,jk,ik->i',X,covariance,X)))
                p0=pred-1.9599639845*errs;p1=pred+1.9599639845*errs
                if SCALES[col]=='log10':
                    pred=10**pred;p0=10**p0;p1=10**p1
                color=COLORS['sex_category'][sex]
                ax.plot(agevalues,pred,lw=2.4,color=color,label=sex)
                ax.fill_between(agevalues,p0,p1,color=color,alpha=.13)
            ax.set(title=f'{bplabel} P{int(q*100)} = {bp_value:.0f} mmHg',xlabel='Age (years)',
                   ylabel=f'Predicted {label} ({unit})')
            if SCALES[col]=='log10':ax.set_yscale('log')
        axes[-1].legend(title='Sex',frameon=True)
        fig.suptitle(f'{label} | four-factor continuous model {stage} | BMI held at median',fontsize=17,y=.995)
        fig.tight_layout(rect=(0,0,1,.95))
        fig.savefig(FIG/slug/'06_continuous'/f'{spec}__adjusted_age_curves.png',dpi=170,bbox_inches='tight')
        plt.close(fig)

# Global association maps keep model families and p-value correction scopes separate.
qmat=hierarchy.pivot(index='hrv_variable',columns='model_stage',values='added_block_q_bh').reindex(
    index=list(HRV),columns=['M1','M2','M3','M4'])
fig,ax=plt.subplots(figsize=(9.4,6))
logq=-np.log10(qmat.clip(lower=1e-12))
annot=qmat.apply(lambda row:[f'{v:.3g}' if np.isfinite(v) else 'N/E' for v in row],axis=1)
labels=np.array(annot.to_list())
sns.heatmap(logq,ax=ax,cmap='YlGnBu',annot=labels,fmt='',linewidths=.7,linecolor='white',
    cbar_kws={'label':'–log10(BH q)'},vmin=0)
ax.set_yticklabels([HRV[h][1] for h in qmat.index],rotation=0)
ax.set(xlabel='Hierarchy block',ylabel='',title='Added interaction block FDR (N/E = unavailable)')
fig.tight_layout()
fig.savefig(GLOBAL/'hierarchical_block_fdr.png',bbox_inches='tight',dpi=170)
plt.close(fig)

fig,axes=plt.subplots(1,2,figsize=(15,5.7))
mat=hierarchy.pivot(index='hrv_variable',columns='model_stage',values='incremental_partial_r2').reindex(
    index=list(HRV),columns=list(STAGES))
sns.heatmap(mat,ax=axes[0],cmap='YlGnBu',annot=True,fmt='.3f',linewidths=.5,
    cbar_kws={'label':'Incremental partial R²'})
axes[0].set(title='Additional in-sample variance explained',ylabel='',xlabel='Stage')
axes[0].set_yticklabels([HRV[h][1] for h in mat.index],rotation=0)
mat2=continuous[continuous.stage=='M4'].pivot(index='hrv_variable',columns='specification',values='block_q_bh_all_continuous').reindex(index=list(HRV))
sns.heatmap(-np.log10(mat2.clip(lower=1e-12)),ax=axes[1],cmap='YlOrBr',annot=mat2.map(lambda z:f'{z:.3g}' if pd.notna(z) else 'N/E'),
    fmt='',linewidths=.5,cbar_kws={'label':'–log10(BH q)'})
axes[1].set(title='Continuous four-way interaction FDR',ylabel='',xlabel='BP measurement')
axes[1].set_yticklabels([HRV[h][1] for h in mat2.index],rotation=0)
axes[1].tick_params(axis='x',rotation=20)
fig.suptitle('Four-factor HRV association and interaction atlas',fontsize=17)
fig.tight_layout(rect=(0,0,1,.945))
fig.savefig(GLOBAL/'four_factor_association_atlas.png',bbox_inches='tight',dpi=170)
plt.close(fig)

fig,ax=plt.subplots(figsize=(10.5,5.7))
tab=stability.pivot(index='hrv_variable',columns='stage',values='bootstrap_valid_fraction').reindex(index=list(HRV),columns=['M2','M3'])
sns.heatmap(tab,ax=ax,cmap='mako',vmin=0,vmax=1,annot=True,fmt='.2f',linewidths=.7,
    cbar_kws={'label':'Full-rank bootstrap fraction'})
ax.set(title='Interaction bootstrap estimability',xlabel='Highest interaction order',ylabel='')
ax.set_yticklabels([HRV[h][1] for h in tab.index],rotation=0)
fig.tight_layout()
fig.savefig(GLOBAL/'bootstrap_estimability.png',bbox_inches='tight',dpi=170)
plt.close(fig)
print('Continuous curves and global synthesis heatmaps generated.')

Continuous curves and global synthesis heatmaps generated.


## 11 — Reproducibility, structural checks and completion inventory

This final cell verifies the complete data cohort, the model and contrast families, complete figure layout, and the readability of **every PNG**. It must run without an error before this notebook is considered validated. Notebook outputs are not edited by subsequent notebooks.

In [12]:
from PIL import Image
assert len(df)==476
assert len(HRV)==7
assert len(support.query("scheme=='original'"))==144
assert len(support.query("scheme=='analysis'"))==36
assert len(hierarchy)==28
assert len(continuous)==56
assert len(three_decomp)==28
assert len(four_decomp)==7
assert len(quantiles)>0
assert len(sensitivity)==7*3*2
assert len(stability)==14
assert len(marginal)==7*sum(len(list(combinations(LEVELS[f],2))) for f in FACTOR_KEYS)
assert len(conditional)==len(nonparam)
assert len(synthesis)==7
assert all(support_summary.total_participants.eq(476))
assert (hierarchy['cv_valid_folds']<=CV_FOLDS).all()
assert (hierarchy['cv_valid_folds']>=0).all()
for col in HRV:
    for order in (1,2,3,4):
        assert len(hierarchy[(hierarchy.hrv_variable==col)&(hierarchy.model_stage==f'M{order}')])==1
for col,(slug,_,_) in HRV.items():
    assert (FIG/slug/'01_descriptive'/'01_original_four_factor_boxplots.png').exists()
    assert len(list((FIG/slug).rglob('*.png')))>=10
images=sorted(FIG.rglob('*.png'))
for path in images:
    with Image.open(path) as im:
        im.verify()
csvs=sorted(TAB.glob('*.csv'))
for path in csvs:
    if pd.read_csv(path).empty:
        raise ValueError(f'Empty table: {path}')
print('09 — Four-Factor HRV Analysis completed')
print(f'Participants: {len(df)} (unmodified original dataset)')
print(f'Categorical hierarchy models: {len(hierarchy)}')
print(f'Continuous hierarchy models: {len(continuous)}')
print(f'Adjusted marginals: {len(marginal)} | conditional effects: {len(conditional)}')
print(f'Three-way decompositions: {len(three_decomp)} | four-way feasibility: {len(four_decomp)}')
print(f'CSV tables: {len(csvs)} | verified PNG figures: {len(images)}')
print('Output directory:',OUT)

09 — Four-Factor HRV Analysis completed
Participants: 476 (unmodified original dataset)
Categorical hierarchy models: 28
Continuous hierarchy models: 56
Adjusted marginals: 56 | conditional effects: 406
Three-way decompositions: 28 | four-way feasibility: 7
CSV tables: 20 | verified PNG figures: 110
Output directory: ../../eda_results/09_four_factor_hrv_analysis
